# Telecom Customer Service Multi-Agent System

LangGraph + LLM (Boeing BCAI or OpenAI) + Embedding Memory

Run cells top to bottom in VS Code (Jupyter extension).

## 1. Imports

Requires: `langgraph`, `langchain`, `langchain-openai`, `faiss-cpu`, `httpx`, `pydantic`

```
pip install langgraph langchain langchain-openai faiss-cpu httpx pydantic
```

In [ ]:
import os
import json
import faiss
import numpy as np
from typing import TypedDict, List, Dict

from langgraph.graph import StateGraph, END


## 2. Provider Toggle

Set `PROVIDER` to `"boeing"` or `"openai"`. This is the only line you need to change to switch backends.

`boeing_chat_model.py` and `boeing_embeddings.py` must sit in the **same folder** as this notebook for the `"boeing"` option to import correctly.

In [ ]:
PROVIDER = "boeing"   # "boeing" or "openai"


## 3. Credentials

Fill in only the credential needed for the `PROVIDER` you selected above. Leaving the unused one blank is fine.

In [ ]:
OPENAI_API_KEY = ""     # required if PROVIDER == "openai"
BOEING_UDAL_PAT = ""    # required if PROVIDER == "boeing"


## 4. Initialize LLM + Embedding Model

Builds `llm` and `embedding_model` for the selected provider, and sets the FAISS vector `dimension` to match the embedding model's output size.

In [ ]:
if PROVIDER == "boeing":
    from boeing_chat_model import BoeingChatModel
    from boeing_embeddings import BoeingEmbeddings

    if not BOEING_UDAL_PAT:
        raise ValueError("PROVIDER is 'boeing' but BOEING_UDAL_PAT is empty. Paste your UDAL_PAT token above.")

    llm = BoeingChatModel(
        udal_pat=BOEING_UDAL_PAT,
        model="gpt-4.1-mini",
        temperature=0,
    )

    embedding_model = BoeingEmbeddings(
        udal_pat=BOEING_UDAL_PAT,
        model="text-embedding-3-large",
    )

    dimension = 3072  # text-embedding-3-large output size

elif PROVIDER == "openai":
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings

    if not OPENAI_API_KEY:
        raise ValueError("PROVIDER is 'openai' but OPENAI_API_KEY is empty. Paste your OpenAI API key above.")

    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
    )

    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small",
    )

    dimension = 1536  # text-embedding-3-small output size

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'boeing' or 'openai'.")

print(f"Using PROVIDER = '{PROVIDER}' | LLM model = '{llm.model}' | Embedding model = '{embedding_model.model}' | dim = {dimension}")


## 5. Memory Store

A simple FAISS `IndexFlatL2` vector store plus parallel Python lists for the raw text and metadata of each memory.

In [ ]:
memory_texts = []
memory_metadata = []

index = faiss.IndexFlatL2(dimension)


### Save memory

In [ ]:
def save_memory(text, metadata={}):
    vector = embedding_model.embed_query(text)

    memory_texts.append(text)
    memory_metadata.append(metadata)

    index.add(np.array([vector]).astype("float32"))


### Search memory

In [ ]:
def search_memory(query, top_k=3):
    if len(memory_texts) == 0:
        return []

    query_vector = embedding_model.embed_query(query)

    D, I = index.search(
        np.array([query_vector]).astype("float32"),
        top_k
    )

    results = []

    for idx in I[0]:
        if idx < len(memory_texts):
            results.append({
                "text": memory_texts[idx],
                "metadata": memory_metadata[idx]
            })

    return results


## 6. Dummy Tools

Stand-ins for real backend calls (billing system, network ops, etc.).

In [ ]:
def create_ticket(issue):
    ticket_id = f"TICK-{1000 + len(memory_texts)}"

    return {
        "ticket_id": ticket_id,
        "status": "OPEN",
        "issue": issue
    }

def check_bill(customer_name):
    return f"{customer_name}'s current bill amount is ₹1299."

def restart_network():
    return "Network restart signal sent successfully."

def check_data_balance():
    return "You have 1.5GB remaining for today."


## 7. Agent State

Shared state schema passed between LangGraph nodes.

In [ ]:
class AgentState(TypedDict):
    user_input: str
    response: str
    agent: str


## 8. Billing Agent — Ravi

Handles recharge, billing, and payment issues.

In [ ]:
def billing_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    result = check_bill("Customer")

    prompt = f"""
You are Ravi, a Telecom Billing Support Agent.

Role:
- Help with recharge
- Billing
- Payment issues

Previous Memories:
{memory_context}

Tool Result:
{result}

Customer Message:
{user_input}

Give a professional customer support reply.
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"Billing Conversation: {user_input} -> {response}"
    )

    return {
        "response": response,
        "agent": "Billing Agent Ravi"
    }


## 9. Technical Support Agent — Priya

Handles internet issues, slow network, router problems. Creates a support ticket.

In [ ]:
def technical_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    network_result = restart_network()

    ticket = create_ticket(user_input)

    prompt = f"""
You are Priya, a Telecom Technical Support Agent.

Role:
- Internet issues
- Slow network
- Router problems

Previous Memories:
{memory_context}

Tool Results:
{network_result}

Generated Ticket:
{ticket}

Customer Message:
{user_input}

Important:
- Mention the ticket ID
- Remember the ticket
- Sound helpful
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"Technical Ticket {ticket['ticket_id']} for issue: {user_input}"
    )

    save_memory(
        f"Technical Conversation: {user_input} -> {response}",
        metadata={"ticket_id": ticket["ticket_id"]}
    )

    return {
        "response": response,
        "agent": "Technical Agent Priya"
    }


## 10. SIM + Network Agent — Arjun

Handles SIM activation, data balance, network coverage, 5G support.

In [ ]:
def sim_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    balance = check_data_balance()

    prompt = f"""
You are Arjun, a Telecom SIM & Network Support Agent.

Role:
- SIM activation
- Data balance
- Network coverage
- 5G support

Previous Memories:
{memory_context}

Tool Result:
{balance}

Customer Message:
{user_input}

Give a useful telecom support response.
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"SIM Conversation: {user_input} -> {response}"
    )

    return {
        "response": response,
        "agent": "SIM Agent Arjun"
    }


## 11. Router

Keyword-based routing to the correct agent node.

In [ ]:
def router(state):
    text = state["user_input"].lower()

    if any(word in text for word in [
        "bill",
        "payment",
        "recharge",
        "invoice"
    ]):
        return "billing"

    elif any(word in text for word in [
        "internet",
        "slow",
        "wifi",
        "network",
        "router",
        "issue",
        "not working"
    ]):
        return "technical"

    else:
        return "sim"


## 12. Build LangGraph

Wires the router and three agent nodes into a compiled graph, `app`.

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("billing", billing_agent)
workflow.add_node("technical", technical_agent)
workflow.add_node("sim", sim_agent)

workflow.set_conditional_entry_point(
    router,
    {
        "billing": "billing",
        "technical": "technical",
        "sim": "sim"
    }
)

workflow.add_edge("billing", END)
workflow.add_edge("technical", END)
workflow.add_edge("sim", END)

app = workflow.compile()


## 13. Interactive Chat Loop

Type a message at the `YOU:` prompt. Type `exit` to stop and print the accumulated memory store.

In [ ]:
print("\n===================================================")
print(" TELECOM CUSTOMER SERVICE MULTI-AGENT SYSTEM ")
print("===================================================")
print("Type \'exit\' to stop.\n")

while True:

    user_query = input("YOU: ")

    if user_query.lower() == "exit":
        print("\nGoodbye 👋")
        break

    result = app.invoke({
        "user_input": user_query
    })

    print("\n[" + result["agent"] + "]")
    print(result["response"])
    print("\n---------------------------------------------------\n")

print("\n================ MEMORY STORE ================\n")

for i, memory in enumerate(memory_texts):
    print(f"{i+1}. {memory}")


## 14. Automated Testing

Runs a fixed set of test queries (billing, technical, memory recall, SIM, multi-turn context), validates memory search, and runs a small stress test.

In [ ]:
test_queries = [

    # ----------------------------------------
    # BILLING TESTS
    # ----------------------------------------

    "My bill amount is too high this month",

    "Recharge was done but not updated",

    "Can you check my payment status?",

    # ----------------------------------------
    # TECHNICAL TESTS
    # ----------------------------------------

    "My internet is not working",

    "WiFi is very slow",

    "Router keeps disconnecting",

    # ----------------------------------------
    # MEMORY TESTS
    # ----------------------------------------

    "What is my ticket ID?",

    "What issue did I report earlier?",

    "Can you summarize my previous complaint?",

    # ----------------------------------------
    # SIM + NETWORK TESTS
    # ----------------------------------------

    "How much data balance do I have?",

    "My SIM card is not activating",

    "Is 5G available in Chennai?",

    # ----------------------------------------
    # MULTI TURN CONTEXT TEST
    # ----------------------------------------

    "I still have the same internet problem",

    "Did you already create a ticket for me?",

    "Can you check previous network issue?"
]

print("\n")
print("=" * 70)
print(" RUNNING TELECOM AGENT TEST CASES ")
print("=" * 70)

for i, query in enumerate(test_queries):

    print(f"\nTEST CASE {i+1}")
    print("-" * 70)

    print(f"\nUSER:")
    print(query)

    result = app.invoke({
        "user_input": query
    })

    print(f"\nAGENT:")
    print(result["agent"])

    print(f"\nRESPONSE:")
    print(result["response"])

    print("\n" + "=" * 70)

print("\n")
print("=" * 70)
print(" MEMORY VALIDATION ")
print("=" * 70)

memory_search_queries = [

    "ticket id",
    "internet issue",
    "billing complaint",
    "SIM issue"
]

for query in memory_search_queries:

    print(f"\nSearching Memory For: {query}")
    print("-" * 50)

    memories = search_memory(query)

    if len(memories) == 0:
        print("No memories found")

    else:
        for idx, mem in enumerate(memories):

            print(f"\nMemory {idx+1}:")
            print(mem["text"])

print("\n")
print("=" * 70)
print(" COMPLETE MEMORY STORE ")
print("=" * 70)

for i, mem in enumerate(memory_texts):

    print(f"\n{i+1}. {mem}")

print("\n")
print("=" * 70)
print(" TICKET RECALL TEST ")
print("=" * 70)

ticket_test = app.invoke({
    "user_input": "Can you tell me my previously generated ticket number?"
})

print(ticket_test["response"])

print("\n")
print("=" * 70)
print(" STRESS TEST ")
print("=" * 70)

stress_queries = [
    "internet problem",
    "bill issue",
    "network slow",
    "payment failed",
    "SIM not working"
]

for i in range(10):

    q = stress_queries[i % len(stress_queries)]

    result = app.invoke({
        "user_input": q
    })

    print(f"\nIteration {i+1}")
    print(f"Query: {q}")
    print("Agent: " + result["agent"])

print("\n")
print("=" * 70)
print(" ALL TESTS COMPLETED ")
print("=" * 70)
